# Experiment 1 Clean Test Notebook Outline

## Purpose
- Validate modularized code in fresh environment (no legacy variables)
- Compare with Shiu et al. (2024) original results
- Serve as template for cloud deployment scripts

---

## Section 1: Environment Setup

### Cell 1: Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from time import time
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import pickle

from caveclient import CAVEclient
from flylif.utils.cave_utils import cave_id, convert_neuron_list_cached

print("Initializing CAVEclient...")
client = CAVEclient('flywire_fafb_public')
print("✅ Connected to FlyWire")


### Cell 2: Module imports

In [ ]:
from flylif.core.parameters import DEFAULT_PARAMS
from flylif.core.data_loader import load_simulation_data
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation
from joblib import Parallel, delayed
from brian2 import Hz

print("✅ Modules imported successfully")

## Section 2: Configuration

### Cell 3: Define paths and neurons


In [ ]:
# Base directory
BASE_DIR = Path.home() / 'mylibrary'/'connectome'/'lifmodel'

# %%
CONFIG = {
    # 基础目录
    'base_dir': BASE_DIR,
    
    # FlyWire 下载的数据文件
    'data_dir': BASE_DIR / 'data_783',
    'connections_file': 'proofread_connections_783.feather',
    'root_ids_file': 'proofread_root_ids_783.npy',
    'classification': 'classification.csv',}


### Cell 4: Test parameters


In [ ]:
# Quick test (for validation)
TEST_MODE = True

if TEST_MODE:
    freq_list = [10, 50, 100, 150, 200]  # 5 freqs
    n_trials = 3
    print("Mode: Quick test (5 freqs × 3 trials)")
else:
    freq_list = list(range(10, 201, 10))  # 19 freqs
    n_trials = 10
    print("Mode: Full parameters (19 freqs × 10 trials)")

print(f"Total simulations: {len(freq_list) * n_trials}")

## Section 2.5: Neuron ID Conversion (v630 → v783)
### Cell 4.5

In [ ]:
# ==================== Cell 新增B: 定义原文v630神经元IDs ====================
print("\n" + "=" * 70)
print("Defining Original Paper Neuron IDs (v630)")
print("=" * 70)

# Original paper neurons (from Shiu et al., FlyWire v630)
NEU_SUGAR_v630 = [
    720575940624963786, 720575940630233916, 720575940637568838, 720575940638202345, 720575940617000768,
    720575940630797113, 720575940632889389, 720575940621754367, 720575940621502051, 720575940640649691,
    720575940639332736, 720575940616885538, 720575940639198653, 720575940620900446, 720575940617937543,
    720575940632425919, 720575940633143833, 720575940612670570, 720575940628853239, 720575940629176663,
    720575940611875570
]

NEU_MN9_v630 = [720575940660219265]  # MN9可能版本间稳定

print(f"\nOriginal neuron counts:")
print(f"  Sugar GRNs: {len(NEU_SUGAR_v630)}")
print(f"  MN9: {len(NEU_MN9_v630)}")

In [ ]:
# ==================== Cell 新增C: 预转换Sugar GRN IDs（缓存）====================
print("\n" + "=" * 70)
print("Converting Sugar GRN IDs (v630 → v783)")
print("=" * 70)
print("⚠️  This uses FlyWire API and may take 1-2 minutes")
print("   Results will be cached to avoid repeated calls\n")

# Cache file
cache_dir = Path('./cache/id_conversions')
cache_dir.mkdir(parents=True, exist_ok=True)
cache_file = cache_dir / 'sugar_grns_v630_to_v783.pkl'

t0 = time()

# Convert with caching
NEU_SUGAR_LEFT = convert_neuron_list_cached(
    old_list=NEU_SUGAR_v630,
    cache_file=cache_file,
    new_ver=783,
    force_update=False,  # Use cache if available
    verbose=True
)

conversion_time = time() - t0

print(f"\n✅ Conversion complete")
print(f"   Time: {conversion_time:.1f}s")
print(f"   Converted IDs: {len(NEU_SUGAR_LEFT)}")

# Show mapping
print(f"\n   ID mapping (first 5):")
for old, new in zip(NEU_SUGAR_v630[:5], NEU_SUGAR_LEFT[:5]):
    match = "✓" if old == new else "✗ CHANGED"
    print(f"   {old} → {new} {match}")

In [ ]:
# ==================== Cell 新增D: 转换MN9（可选）====================
print("\nConverting MN9 IDs...")

cache_file_mn9 = cache_dir / 'mn9_v630_to_v783.pkl'

NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=NEU_MN9_v630,
    cache_file=cache_file_mn9,
    new_ver=783,
    verbose=True
)

print(f"   MN9: {NEU_MN9_v630[0]} → {NEU_MN9_RIGHT[0]}")

# ==================== Cell 新增E: 保存转换结果（备份）====================
# 保存转换后的神经元列表供后续使用
neuron_ids_v783 = {
    'NEU_SUGAR_LEFT': NEU_SUGAR_LEFT,
    'NEU_MN9_RIGHT': NEU_MN9_RIGHT,
    'conversion_time': conversion_time,
    'source_version': 630,
    'target_version': 783,
}

backup_file = cache_dir / 'neuron_ids_v783.pkl'
with open(backup_file, 'wb') as f:
    pickle.dump(neuron_ids_v783, f)

print(f"\n✅ Neuron IDs ready for simulation")
print(f"   Backup saved: {backup_file}")
print(f"\n" + "=" * 70)
print("⚠️  Important: Use NEU_SUGAR_LEFT (v783) for simulation")
print("=" * 70)

## Section 3: Data Loading
### Cell 5: Load optimized data

In [ ]:
print("Loading data...")
t0 = time()

DATA = load_simulation_data(CONFIG)

print(f"✅ Loaded in {time()-t0:.1f}s")
print(f"   Data size: {DATA['df_conn'].memory_usage(deep=True).sum()/1e6:.0f} MB")

## Section 4: Parallel Execution

### Cell 6: Worker function

In [ ]:
# ==================== Cell 6: Worker Function ====================
def run_freq_worker(freq, neu_exc, data, params, n_trials):
    """Worker function for parallel execution."""
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    from brian2 import Hz
    
    # Build network
    columns = data['columns']
    net_components = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    # Run simulation
    result = run_simulation(
        net_components=net_components,
        neu_exc=neu_exc,
        params={'r_poi': freq * Hz},
        n_trials=n_trials,
        verbose=False
    )
    
    return (freq, {
        'n_active': result['n_active'],
        'n_spikes': result['n_spikes'],
        'df': result['df'].copy(),
    })

### Cell 7: Run parallel simulation
```python

In [ ]:
# ==================== Cell 7: Run Parallel ====================
print("=" * 70)
print("Experiment 1: Sugar GRN Frequency Sweep")
print("=" * 70)

print(f"\nConfiguration:")
print(f"  Frequencies: {len(freq_list)} points {freq_list}")
print(f"  Trials/freq: {n_trials}")
print(f"  Workers: 6")
print(f"  Data size: 68 MB")

# Prepare tasks
tasks = [(freq, NEU_SUGAR_LEFT, DATA, DEFAULT_PARAMS, n_trials) 
         for freq in freq_list]

print(f"\nStarting parallel execution...")
t0 = time()

results = Parallel(n_jobs=6, verbose=10)(
    delayed(run_freq_worker)(*task) for task in tasks
)

total_time = time() - t0

# Organize results
results_dict = dict(results)

print(f"\n" + "=" * 70)
print(f"✅ Complete!")
print("=" * 70)
print(f"Total time: {total_time/60:.2f} minutes")
print(f"Time/frequency: {total_time/len(freq_list):.1f} seconds")

## Section 5: Results Processing

### Cell 8: Calculate firing rates

In [ ]:

# Extract firing rates for all neurons
all_firing_rates = {}
duration_s = 1.0  # 1000ms

for freq, res in results_dict.items():
    df = res['df']
    if len(df) > 0:
        counts = df.groupby('flywire_id').size()
        rates = counts / (n_trials * duration_s)
        all_firing_rates[freq] = rates.to_dict()
    else:
        all_firing_rates[freq] = {}


### Cell 9: Summary statistics

In [ ]:
# Create summary table
summary = []
for freq in sorted(freq_list):
    summary.append({
        'Frequency': freq,
        'Active Neurons': results_dict[freq]['n_active'],
        'Total Spikes': results_dict[freq]['n_spikes'],
    })

df_summary = pd.DataFrame(summary)
print(df_summary)

## Section 6: Comparison with Original

### Cell 10: Load Original Data

In [ ]:
# ==================== Cell 10: Load Original Data ====================
import pyarrow.parquet as pq

print("\n" + "=" * 70)
print("Loading Original Data (Shiu et al. 2024)")
print("=" * 70)

# Original data path
orig_data_path = BASE_DIR / 'Drosophila_brain_model/results/example/sugarR_100Hz.parquet'

if not orig_data_path.exists():
    print(f"\n⚠️  Original data not found: {orig_data_path}")
    print(f"   Skipping comparison")
    has_original = False
else:
    print(f"\nLoading from: {orig_data_path}")
    
    # Load parquet (column-by-column)
    pf = pq.ParquetFile(orig_data_path)
    df_orig = pd.DataFrame({
        't': pf.read(['t']).column('t').to_pylist(),
        'trial': pf.read(['trial']).column('trial').to_pylist(),
        'flywire_id': pf.read(['flywire_id']).column('flywire_id').to_pylist(),
    })
    
    print(f"\n✅ Original data loaded:")
    print(f"   Spikes: {len(df_orig):,}")
    print(f"   Active neurons: {df_orig['flywire_id'].nunique()}")
    print(f"   Parameters: 30 trials × 1000ms")
    
    # Calculate firing rates (原文)
    spike_counts_orig = df_orig.groupby('flywire_id').size()
    rate_orig = spike_counts_orig / (30 * 1.0)  # 30 trials, 1 second
    rate_orig.name = 'Original'
    
    has_original = True
    
    print(f"\n   Firing rate statistics (Original):")
    print(f"   Mean: {rate_orig.mean():.2f} Hz")
    print(f"   Max: {rate_orig.max():.2f} Hz")
    print(f"   Active (>0): {(rate_orig > 0).sum()}")

### Cell 11: Calculate correlation

In [ ]:
# ==================== Cell 11: Calculate Correlation ====================
if has_original:
    print("\n" + "=" * 70)
    print("Correlation Analysis (100 Hz)")
    print("=" * 70)
    
    # Our results (100Hz)
    rate_ours = pd.Series(all_firing_rates[100], name='Ours')
    
    # Merge for comparison
    df_compare = pd.concat([rate_orig, rate_ours], axis=1).fillna(0)
    
    # Calculate correlation
    r, p = pearsonr(df_compare['Original'], df_compare['Ours'])
    
    print(f"\nResults:")
    print(f"  Original (30 trials): {len(rate_orig)} active neurons")
    print(f"  Ours ({n_trials} trials):     {(rate_ours>0).sum()} active neurons")
    print(f"  Common active:        {((df_compare['Original']>0) & (df_compare['Ours']>0)).sum()}")
    
    print(f"\nCorrelation:")
    print(f"  Pearson r = {r:.4f}")
    print(f"  p-value = {p:.2e}")
    
    if r > 0.80:
        print(f"  ✅ Excellent correlation (r > 0.80)")
    elif r > 0.70:
        print(f"  ✓ Good correlation (r > 0.70)")
    elif r > 0.60:
        print(f"  ⚠️ Fair correlation (consider more trials)")
    else:
        print(f"  ❌ Low correlation (needs investigation)")
    
    # Top neurons overlap
    top_n = 50
    top_orig = rate_orig.nlargest(top_n).index
    top_ours = rate_ours.nlargest(top_n).index
    overlap = len(set(top_orig) & set(top_ours))
    
    print(f"\nTop {top_n} neurons overlap:")
    print(f"  Common: {overlap}/{top_n} ({100*overlap/top_n:.0f}%)")
    
else:
    print("\n⚠️ Original data not available, skipping comparison")
    df_compare = None

## Section 6.5: Discrepancy Analysis with ID Conversion
### Cell 11.5

In [ ]:
# =====================================================================

# ==================== Cell 新增F: 识别差异最大的神经元 ====================
if has_original and 'df_compare' in dir():
    print("\n" + "=" * 70)
    print("Discrepancy Analysis: Top Mismatch Neurons")
    print("=" * 70)
    
    # Calculate residuals (only for active neurons in both)
    active_both = (df_compare['Original'] > 0) & (df_compare['Ours'] > 0)
    df_active = df_compare[active_both].copy()
    
    # Calculate absolute difference
    df_active['abs_diff'] = np.abs(df_active['Original'] - df_active['Ours'])
    df_active['rel_diff'] = df_active['abs_diff'] / df_active['Original']
    
    # Top 50 largest discrepancies
    top_50_mismatch = df_active.nlargest(50, 'abs_diff')
    
    print(f"\nTop 50 neurons with largest firing rate discrepancies:")
    print(f"  Mean abs diff: {top_50_mismatch['abs_diff'].mean():.2f} Hz")
    print(f"  Mean rel diff: {top_50_mismatch['rel_diff'].mean()*100:.1f}%")
    
    print(f"\nTop 10 examples:")
    print(top_50_mismatch[['Original', 'Ours', 'abs_diff']].head(10).to_string())
    
    # Extract IDs (these are v630 from original paper)
    mismatch_ids_v630 = top_50_mismatch.index.tolist()
    
    print(f"\n✅ Identified {len(mismatch_ids_v630)} neuron IDs with high discrepancy")
else:
    print("⚠️ Skipping discrepancy analysis (no original data)")


# ==================== Cell 新增G: 转换差异神经元IDs ====================
if has_original and 'mismatch_ids_v630' in dir():
    print("\n" + "=" * 70)
    print("Converting Mismatch Neuron IDs (v630 → v783)")
    print("=" * 70)
    print("Purpose: Check if discrepancies due to ID version mismatch\n")
    
    cache_file_mismatch = cache_dir / 'mismatch_neurons_v630_to_v783.pkl'
    
    # Convert top 50 mismatch IDs
    t0 = time()
    
    mismatch_ids_v783 = convert_neuron_list_cached(
        old_list=mismatch_ids_v630,
        cache_file=cache_file_mismatch,
        new_ver=783,
        force_update=False,
        verbose=True
    )
    
    print(f"\n✅ Conversion complete ({time()-t0:.1f}s)")
    
    # Create mapping
    id_mapping = {old: new for old, new in zip(mismatch_ids_v630, mismatch_ids_v783)}
    
    # Check how many IDs changed
    n_changed = sum(1 for old, new in id_mapping.items() if old != new)
    
    print(f"\nID version changes:")
    print(f"  Changed: {n_changed}/{len(id_mapping)} ({100*n_changed/len(id_mapping):.0f}%)")
    print(f"  Unchanged: {len(id_mapping) - n_changed}")
    
    # Show changed IDs
    if n_changed > 0:
        print(f"\nChanged IDs (first 10):")
        count = 0
        for old, new in id_mapping.items():
            if old != new and count < 10:
                print(f"   {old} → {new}")
                count += 1


# ==================== Cell 新增H: 重新对比（使用v783 IDs）====================
if has_original and 'mismatch_ids_v783' in dir():
    print("\n" + "=" * 70)
    print("Re-analysis: Using Converted v783 IDs")
    print("=" * 70)
    
    # Remap original data: v630 IDs → v783 IDs
    print("\nRemapping original data to v783 IDs...")
    
    df_orig_remapped = df_orig.copy()
    df_orig_remapped['flywire_id'] = df_orig_remapped['flywire_id'].map(
        lambda x: id_mapping.get(x, x)  # Use mapping if available, else keep original
    )
    
    # Recalculate firing rates with v783 IDs
    spike_counts_remapped = df_orig_remapped.groupby('flywire_id').size()
    rate_orig_remapped = spike_counts_remapped / 30.0
    rate_orig_remapped.name = 'Original_v783'
    
    # Our results (already in v783)
    rate_ours = pd.Series(all_firing_rates[100], name='Ours_v783')
    
    # New comparison
    df_compare_v783 = pd.concat([rate_orig_remapped, rate_ours], axis=1).fillna(0)
    
    # Recalculate correlation
    r_v783, p_v783 = pearsonr(df_compare_v783['Original_v783'], df_compare_v783['Ours_v783'])
    
    print(f"\nCorrelation comparison:")
    print(f"  Before ID conversion (v630 vs v783): r = {r:.4f}")
    print(f"  After ID conversion (v783 vs v783):  r = {r_v783:.4f}")
    print(f"  Improvement: {r_v783 - r:+.4f}")
    
    if r_v783 > r + 0.01:
        print(f"  ✅ ID conversion improved correlation!")
    elif abs(r_v783 - r) < 0.01:
        print(f"  ✓ ID version has minimal impact")
    else:
        print(f"  ⚠️ Unexpected: correlation decreased")
    
    # Check mismatch neurons specifically
    print(f"\nFocus on top 50 mismatch neurons:")
    
    # Find how many are now better matched
    better_matched = 0
    for neuron_id in mismatch_ids_v783:
        if neuron_id in df_compare_v783.index:
            old_diff = top_50_mismatch.loc[
                id_mapping.get(neuron_id, neuron_id), 'abs_diff'
            ] if id_mapping.get(neuron_id, neuron_id) in top_50_mismatch.index else np.inf
            
            new_orig = df_compare_v783.loc[neuron_id, 'Original_v783']
            new_ours = df_compare_v783.loc[neuron_id, 'Ours_v783']
            new_diff = abs(new_orig - new_ours)
            
            if new_diff < old_diff * 0.8:  # 20% improvement
                better_matched += 1
    
    print(f"  Improved: {better_matched}/{len(mismatch_ids_v783)}")
    
    # Update global comparison for visualization
    df_compare = df_compare_v783
    r = r_v783
    
    print(f"\n✅ Using v783-corrected comparison for subsequent analysis")

else:
    print("\n⚠️ Skipping ID conversion (no original data or mismatch IDs)")

### Cell 12: Visualization

In [ ]:
# ==================== Cell 12: Visualization ====================
if has_original and df_compare is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Plot 1: Correlation scatter
    ax = axes[0]
    ax.scatter(df_compare['Original_v783'], df_compare['Ours_v783'], alpha=0.3, s=20)
    max_val = df_compare.max().max()
    ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='y = x')
    ax.set_xlabel('Original Firing Rate (Hz)', fontsize=11)
    ax.set_ylabel('Our Firing Rate (Hz)', fontsize=11)
    ax.set_title(f'100Hz Firing Rate Correlation\n(r = {r:.3f}, n_trials = {n_trials})', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Response spectrum
    ax = axes[1]
    for freq in sorted(freq_list):
        rates = list(all_firing_rates[freq].values())
        if rates:
            ax.hist(rates, bins=30, alpha=0.5, label=f'{freq} Hz', edgecolor='none')
    ax.set_xlabel('Firing Rate (Hz)', fontsize=11)
    ax.set_ylabel('Neuron Count', fontsize=11)
    ax.set_title('Response Spectrum Across Frequencies', fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
    
else:
    print("Skipping visualization (no original data)")

## Section 7: Save Results

### Cell 13: Export

In [ ]:
# ==================== Cell 13: Save Results ====================
output_dir = Path('./results/exp1_clean_test_10trails')
output_dir.mkdir(parents=True, exist_ok=True)

print(f"\nSaving results to: {output_dir}")

# Save complete results
with open(output_dir / 'results.pkl', 'wb') as f:
    pickle.dump({
        'results_dict': results_dict,
        'all_firing_rates': all_firing_rates,
        'parameters': {
            'freq_list': freq_list,
            'n_trials': n_trials,
            'n_workers': 6,
            'activated_neurons': NEU_SUGAR_LEFT,
            'total_time_min': total_time / 60,
        },
        'performance': {
            'total_time_sec': total_time,
            'time_per_freq': total_time / len(freq_list),
        }
    }, f)

print(f"  ✓ results.pkl")

# Save summary table
df_summary.to_csv(output_dir / 'summary.csv', index=False)
print(f"  ✓ summary.csv")

# Save comparison data (if available)
if has_original and 'df_compare' in dir():
    df_compare.to_csv(output_dir / 'comparison_100Hz.csv')
    print(f"  ✓ comparison_100Hz.csv")
    
    # Save correlation stats
    with open(output_dir / 'correlation.txt', 'w') as f:
        f.write(f"Pearson r: {r:.4f}\n")
        f.write(f"p-value: {p:.2e}\n")
        f.write(f"Original neurons: {len(rate_orig)}\n")
        f.write(f"Our neurons ({n_trials} trials): {(rate_ours>0).sum()}\n")
    print(f"  ✓ correlation.txt")

print(f"\n✅ All results saved to: {output_dir}")